# Part 1: The Stage (Autonomous Agents)

This notebook runs the **Autonomous Agents**. 
It polls the Fluss `chatroom` table and prints everything that happens in the session.

## 0. Reset (Run this to start from scratch)

In [ ]:
import sys

# 1. Install maturin specifically into the Notebook's Python environment
!{sys.executable} -m pip install maturin

# 2. Compile and install your local fluss-rust code into this environment
!cd /Users/jaredyu/Desktop/open_source/fluss-rust/bindings/python && {sys.executable} -m maturin develop

In [12]:
import fluss
import os
from datetime import datetime

# Get the exact path of the loaded compiled _fluss extension
extension_path = fluss._fluss.__file__
mod_time = os.path.getmtime(extension_path)

print(f"Loaded Extension: {extension_path}")
print(f"Compiled At: {datetime.fromtimestamp(mod_time)}")

Loaded Extension: /Users/jaredyu/Desktop/open_source/fluss-rust/bindings/python/fluss/_fluss.cpython-312-darwin.so
Compiled At: 2026-03-14 18:45:35.589391


In [13]:
DEFAULT_MODEL = "gemini-3-flash-preview"

In [14]:
import fluss
import pyarrow as pa
import asyncio

async def reset_chatroom():
    config = fluss.Config({"bootstrap.servers": "127.0.0.1:9123"})
    conn = await fluss.FlussConnection.create(config)
    admin = await conn.get_admin()
    table_path = fluss.TablePath("containerclaw", "chatroom")
    
    print("Wiping chatroom...")
    await admin.drop_table(table_path, ignore_if_not_exists=True)
    
    schema = fluss.Schema(pa.schema([
        pa.field("ts", pa.int64()), 
        pa.field("actor_id", pa.string()), 
        pa.field("content", pa.string())
    ]))
    await admin.create_table(table_path, fluss.TableDescriptor(schema, bucket_count=1))
    print("🧹 Chatroom wiped and reset.")
    conn.close()

await reset_chatroom()

Wiping chatroom...
🧹 Chatroom wiped and reset.


## 1. Init & Setup

In [15]:
import fluss
import os
import asyncio
import pyarrow as pa
import pandas as pd
from datetime import datetime
import time
import requests
from dotenv import load_dotenv

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    with open("../containerclaw/secrets/gemini_api_key.txt", "r") as f:
        GEMINI_API_KEY = f.read().strip()

async def connect_stage():
    config = fluss.Config({"bootstrap.servers": "127.0.0.1:9123"})
    conn = await fluss.FlussConnection.create(config)
    table_path = fluss.TablePath("containerclaw", "chatroom")
    return conn, table_path

conn, table_path = await connect_stage()
print("✅ Stage connected to Fluss.")

✅ Stage connected to Fluss.


## 2. Gemini Agent Loop

In [16]:
import random
import json

class GeminiAgent:
    def __init__(self, agent_id, persona):
        self.agent_id = agent_id
        self.persona = persona
        self.model = DEFAULT_MODEL

    def _format_history(self, raw_messages):
        """Tailors the history for this specific agent's perspective."""
        formatted = []
        for msg in raw_messages:
            actor = msg['actor_id']
            content = msg['content']
            
            # If I sent it, role is "model". If anyone else (Human, Agent, or Moderator) sent it, "user".
            role = "model" if actor == self.agent_id else "user"
            
            # Formatting for the prompt
            if actor == "Moderator":
                text = f"[Moderator Note]: {content}"
            elif role == "user":
                text = f"{actor}: {content}"
            else:
                text = content # Model role doesn't need prefix
            
            formatted.append({"role": role, "parts": [{"text": text}]})
        return formatted

    async def _vote(self, raw_messages, candidates, previous_votes=None):
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{self.model}:generateContent?key={GEMINI_API_KEY}"
        
        contents = self._format_history(raw_messages)
        
        system_instruction = (
            f"You are {self.agent_id}. Persona: {self.persona}.\n"
            f"You are in a voting phase. A new message has arrived in the chat.\n"
            f"You must review the history and vote for the ONE agent who is best suited to respond.\n"
            f"Candidates: {candidates}.\n"
            "If someone specifically addressed an agent, vote for them. Otherwise, vote based on merit.\n"
            "Respond ONLY in valid JSON with 'vote' (name) and 'reason' (one sentence)."
        )

        if previous_votes:
            system_instruction += (
                "\n\n### DEBATE MODE ###\n"
                f"Previous round results:\n{previous_votes}\n"
                "You are in a tie-breaker round. Read the reasoning from other agents above. "
                "Acknowledge their points. You must now either defend your original choice with stronger logic or "
                "concede and vote for another agent if their reasoning was more compelling. "
                "We must reach a consensus."
            )

        payload = {
            "system_instruction": {"parts": [{"text": system_instruction}]},
            "contents": contents,
            "generationConfig": {
                "response_mime_type": "application/json"
            }
        }
        
        try:
            res = requests.post(url, json=payload, timeout=60)
            if res.status_code == 200:
                data = res.json()
                raw_text = data['candidates'][0]['content']['parts'][0]['text']
                return json.loads(raw_text)
            else:
                print(f"❌ [{self.agent_id}] Voting API Error {res.status_code}: {res.text}")
                return None
        except Exception as e:
            print(f"❌ [{self.agent_id}] Voting failed: {e}")
            return None

    async def _think(self, raw_messages):
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{self.model}:generateContent?key={GEMINI_API_KEY}"
        
        contents = self._format_history(raw_messages)
        
        system_instruction = (
            f"You are {self.agent_id}, participating in a multi-agent chat. "
            f"Persona: {self.persona}. "
            "Respond to the conversation if appropriate. "
            "If no action is needed or you just spoke, respond with [WAIT].\n\n"
            "CRITICAL: If the Moderator just announced you won the election, you SHOULD contribute. "
            "If you are waiting for someone else to finish research, acknowledge it and explain what you expect from them. "
            "Do not just [WAIT] if you were specifically chosen to speak."
        )
        payload = {
            "system_instruction": {"parts": [{"text": system_instruction}]},
            "contents": contents
        }
        try:
            res = requests.post(url, json=payload, timeout=60)
            if res.status_code == 200:
                data = res.json()
                return data['candidates'][0]['content']['parts'][0]['text'].strip()
            else:
                print(f"❌ [{self.agent_id}] Thinking API Error {res.status_code}: {res.text}")
                return None
        except Exception as e:
            print(f"❌ [{self.agent_id}] Thinking failed: {e}")
            return None

class StageModerator:
    def __init__(self, table):
        self.table = table
        self.history_keys = set()
        self.all_messages = [] # PERSISTENT HISTORY across poll cycles
        self.writer = table.new_append().create_writer()

    async def run(self, agents):
        scanner = await self.table.new_scan().create_record_batch_log_scanner()
        scanner.subscribe(bucket_id=0, start_offset=0)
        
        agent_names = [a.agent_id for a in agents]
        print(f"⚖️ [Moderator] Active with agents: {agent_names}")
        
        while True:
            poll = scanner.poll_arrow(timeout_ms=500)
            if poll.num_rows > 0:
                df = poll.to_pandas()
                new_msg_detected = False
                
                # Detect and store new messages
                for _, row in df.iterrows():
                    key = f"{row['ts']}-{row['actor_id']}"
                    if key not in self.history_keys:
                        self.history_keys.add(key)
                        msg_obj = {"actor_id": row['actor_id'], "content": row['content']}
                        self.all_messages.append(msg_obj)
                        
                        if row['actor_id'] == "Human":
                            new_msg_detected = True
                            print(f"📢 [Human said]: {row['content']}")
                        elif row['actor_id'] in agent_names:
                            print(f"👂 [Heard] [{row['actor_id']}]: {row['content']}")

                if new_msg_detected:
                    await asyncio.sleep(1.0) 
                    
                    # Pass the full HISTORY to the voters
                    context_window = self.all_messages[-20:]
                    
                    winner, election_log = await self.elect_leader(agents, context_window, agent_names)
                    
                    # INJECT ELECTION LOG INTO HISTORY FOR AGENTS TO SEE
                    self.all_messages.append({"actor_id": "Moderator", "content": f"Election Summary:\n{election_log}"})
                    
                    if winner:
                        winning_agent = next(a for a in agents if a.agent_id == winner)
                        print(f"🧠 [Moderator] {winner} won the election. Executing...")
                        
                        # Pass the updated history (including the moderator note) to the winner
                        updated_context = self.all_messages[-20:]
                        resp = await winning_agent._think(updated_context)
                        
                        if resp and "[WAIT]" not in resp:
                            print(f"📢 [{winner} says]: {resp}")
                            await self.publish(winner, resp)
                        else:
                            print(f"💤 [{winner}] chose to WAIT or failed to respond. Nudging...")
                            
                            # AUTOMATED NUDGE
                            nudge_text = f"@{winner}, you won the election but chose to WAIT. Could you briefly explain why so the team knows what you're waiting for?"
                            self.all_messages.append({"actor_id": "Moderator", "content": nudge_text})
                            
                            # Second attempt with nudge context
                            nudge_context = self.all_messages[-20:]
                            resp = await winning_agent._think(nudge_context)
                            
                            if resp:
                                print(f"📢 [{winner} explanation]: {resp}")
                                await self.publish(winner, resp)
                            else:
                                print(f"❌ [{winner}] remains silent after nudge.")
            
            await asyncio.sleep(1)

    async def elect_leader(self, agents, history, agent_names):
        previous_votes_context = None
        election_log_collector = []
        
        for r in range(1, 4):
            election_log_collector.append(f"--- Round {r} ---")
            print(f"🗳️ [Moderator] Election Round {r} starting...")
            votes = await asyncio.gather(*[a._vote(history, agent_names, previous_votes_context) for a in agents])
            
            tally = {}
            attribution_list = []
            
            for agent, vote_result in zip(agents, votes):
                if vote_result and "vote" in vote_result:
                    nominee = vote_result['vote']
                    reason = vote_result.get('reason', 'N/A')
                    tally[nominee] = tally.get(nominee, 0) + 1
                    vote_str = f"{agent.agent_id} voted for {nominee}. Reason: '{reason}'"
                    attribution_list.append(vote_str)
                    election_log_collector.append(vote_str)
                    print(f"🗣️ [{agent.agent_id}] voted for {nominee} -> \"{reason}\"")
                else:
                    print(f"⚠️ [{agent.agent_id}] failed to cast a valid vote.")

            if not tally:
                return random.choice(agent_names), "No valid votes received."

            tally_str = f"Tally: {tally}"
            election_log_collector.append(tally_str)
            print(f"📊 [Moderator] Round {r} {tally_str}")
            
            max_votes = max(tally.values())
            winners = [name for name, count in tally.items() if count == max_votes]
            
            if len(winners) == 1:
                return winners[0], "\n".join(election_log_collector)
            
            previous_votes_context = " | ".join(attribution_list)
            print(f"⚖️ [Moderator] Round {r} ended in a tie: {winners}")
        
        choice = random.choice(winners)
        election_log_collector.append(f"Tie persists. Circuit breaker chose: {choice}")
        print(f"🎲 [Moderator] Tie persists. Circuit breaker choosing: {choice}")
        return choice, "\n".join(election_log_collector)

    async def publish(self, actor_id, content):
        batch = pa.RecordBatch.from_arrays([
            pa.array([int(time.time() * 1000)], type=pa.int64()),
            pa.array([actor_id], type=pa.string()),
            pa.array([content], type=pa.string())
        ], schema=pa.schema([
            pa.field("ts", pa.int64()), 
            pa.field("actor_id", pa.string()), 
            pa.field("content", pa.string())
        ]))
        self.writer.write_arrow_batch(batch)
        await self.writer.flush()

In [17]:
table = await conn.get_table(table_path)
alice = GeminiAgent("Alice", "Software architect.")
bob = GeminiAgent("Bob", "Project manager.")
carol = GeminiAgent("Carol", "Software engineer.")
david = GeminiAgent("David", "Software QA tester.")
eve = GeminiAgent("Eve", "Business user.")
moderator = StageModerator(table)

print("--- STAGE ACTIVE (Democratic Moderator) ---")
await moderator.run([alice, bob, carol, david, eve])

--- STAGE ACTIVE (Democratic Moderator) ---
⚖️ [Moderator] Active with agents: ['Alice', 'Bob', 'Carol', 'David', 'Eve']
📢 [Human said]: create a tic tac toe game
🗳️ [Moderator] Election Round 1 starting...
🗣️ [Alice] voted for Alice -> "As a software architect, I am best suited to design the structure and provide the initial implementation of the Tic Tac Toe game."
🗣️ [Bob] voted for Alice -> "As the developer, Alice is best equipped to handle the initial implementation and coding of the tic-tac-toe game."
🗣️ [Carol] voted for Carol -> "As a software engineer, I am best suited to implement the game logic and provide the source code for the requested Tic Tac Toe game."
🗣️ [David] voted for Alice -> "Alice, being a software developer, is the most appropriate person to write the actual code for a tic-tac-toe game."
🗣️ [Eve] voted for Alice -> "Alice is typically the lead developer or architect best suited to handle the technical implementation of a coding project."
📊 [Moderator] Round 1 

CancelledError: 

In [ ]:
# import json

# # Get the last 20 messages just like the moderator does
# context_window = moderator.all_messages[-20:]

# # Format it through Alice's perspective
# alice_perspective = alice._format_history(context_window)

# # Print it nicely formatted
# print(json.dumps(alice_perspective, indent=2))

In [ ]:
# 1. Connect
config = fluss.Config({"bootstrap.servers": "127.0.0.1:9123"})
conn = await fluss.FlussConnection.create(config)
table_path = fluss.TablePath("containerclaw", "chatroom")
table = await conn.get_table(table_path)

# 2. Scan all rows from the beginning
scanner = await table.new_scan().create_record_batch_log_scanner()
scanner.subscribe(bucket_id=0, start_offset=0)

# 3. Poll and convert to DataFrame
poll = scanner.poll_arrow(timeout_ms=1000)
if poll.num_rows > 0:
    df = poll.to_pandas()
    # Sort by timestamp to see the conversation flow
    print(df.sort_values('ts'))
else:
    print("Table is empty.")

conn.close()